<a href="https://colab.research.google.com/github/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/blob/main/notebooks/Aula05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://keystoneacademic-res.cloudinary.com/image/upload/c_pad,w_640,h_304/dpr_auto/f_auto/q_auto/v1/element/94/94774_thumb.png" width=300>

# Programação para Análise de Dados

## Aula 5: Variáveis e suas características

### Professor: Carlos E. Leal de Castro

## Agenda
1. Conceitos básicos: o que é variável e como classificamos  
2. Tipos de variáveis: **numéricas** (contínuas/discretas), **categóricas** (nominais/ordinais), **binárias**  
3. **Manipulação de data/hora** com `pandas` (`datetime`)  
4. **Criação e transformação** de variáveis (engenharia de dados)  
5. **Conversão entre tipos** (`astype`, `to_datetime`, `to_numeric`, `Categorical`)  
6. Exercícios práticos ao longo e **lista final** para consolidar

## Objetivos de Aprendizagem
Ao final, você será capaz de:
- Identificar e classificar variáveis (numéricas, categóricas, ordinais, binárias, datas).
- Criar, transformar e converter tipos de dados de forma segura no `pandas`.
- Lidar com datas/horas: parsing, operações temporais e *feature engineering* de tempo.
- Implementar boas práticas de limpeza e tipagem para análises e modelagem.

## Preparação do Ambiente
Execute a célula abaixo para carregar as bibliotecas usadas nesta aula.

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
print("Tudo pronto!")

Tudo pronto!


## 1) Conceitos: Variável e Medição
- **Variável**: característica mensurável de uma unidade (pessoa, produto, transação, etc.).  
- **Domínio** (ou *range*): conjunto de valores possíveis.  
- **Escala**: nominal, ordinal, intervalar, razão (estatística/medidas).
- **No `pandas`**, variáveis são colunas de um `DataFrame`; cada coluna tem um **dtype**.

> **Dica**: escolha o dtype correto o quanto antes — melhora performance, memória e evita bugs.

In [13]:
# DataFrame de exemplo com diversos tipos
df_demo = pd.DataFrame({
    "produto": ["A", "B", "C"],
    "categoria": pd.Categorical(["bronze", "prata", "ouro"], ordered=True),
    "qtd": [10, 5, 8],
    "preco": [12.5, 20.0, 12.5],
    "ativo": [True, True, False],
    "data_venda": pd.to_datetime(["2025-08-10","2025-08-11","2025-08-12"]),
})
display(df_demo)
display(df_demo.dtypes)

,produto,categoria,qtd,preco,ativo,data_venda
0,A,bronze,10,12.5,True,2025-08-10
1,B,prata,5,20.0,True,2025-08-11
2,C,ouro,8,12.5,False,2025-08-12


,0
produto,object
categoria,category
qtd,int64
preco,float64
ativo,bool
data_venda,datetime64[ns]


### Exercício Prático 1
Crie um `DataFrame` com ao menos 5 colunas representando:
- 2 numéricas (int, float),
- 1 categóricas (use `pd.Categorical(ordered=True)`),
- 1 binária (booleana),
- 1 de data (`datetime` - use `pd.to_datetime()`).  
Depois, mostre `dtypes` e explique (em comentário) a escolha de cada tipo.

## 2) Variáveis Numéricas: Discretas vs Contínuas
- **Discretas**: contagem (inteiros: 0, 1, 2…). Ex.: número de itens, cliques.
- **Contínuas**: medição em escala contínua. Ex.: preço, tempo, temperatura.
- Operações comuns: soma, média, mediana, desvio-padrão, normalização, *winsorization*.

In [14]:
# Estatísticas descritivas e operações numéricas
df_num = df_demo[["qtd","preco"]].copy()
print(df_num.describe())

# Criando métricas
df_num["receita"] = df_demo["qtd"] * df_demo["preco"]
df_num["preco_log"] = np.log(df_demo["preco"])
display(df_num.head())

             qtd      preco
count   3.000000   3.000000
mean    7.666667  15.000000
std     2.516611   4.330127
min     5.000000  12.500000
25%     6.500000  12.500000
50%     8.000000  12.500000
75%     9.000000  16.250000
max    10.000000  20.000000


,qtd,preco,receita,preco_log
0,10,12.5,125.0,2.525729
1,5,20.0,100.0,2.995732
2,8,12.5,100.0,2.525729


### Exercício Prático 2
Dado o `df_num`, crie:
1. A coluna `ticket_medio` = receita por item (`receita / qtd`).  
2. Uma coluna `qtd_zscore` com padronização $ z = \frac{x - \mu}{\sigma}$.  

In [15]:
df_num['ticket_medio'] = df_num['receita'] / df_num['qtd']

media = df_num['ticket_medio'].mean()
std = df_num['ticket_medio'].std()

df_num['qtd_zscore']  = (df_num['ticket_medio'] - media) / std

df_num.head()

,qtd,preco,receita,preco_log,ticket_medio,qtd_zscore
0,10,12.5,125.0,2.525729,12.5,-0.577350
1,5,20.0,100.0,2.995732,20.0,1.154701
2,8,12.5,100.0,2.525729,12.5,-0.577350


## 3) Variáveis Binárias (Booleanas)
- Representam dois estados: `True/False` (ou 0/1).  
- Cuidados: `object` com "Sim/Não" deve virar `bool`; coerção com `map`/`replace`.
- Úteis para filtros, *flags* de qualidade, *targets* binários em ML.

In [16]:
serie_bin = pd.Series(["Sim","Não","Sim","Não","Sim"])
# Converter para bool (Sim=True, Não=False) usando map
bin_bool = serie_bin.map({"Sim": True, "Não": False})
display(bin_bool)
display(bin_bool.dtype)

# Exemplo com 0/1
bin_int = bin_bool.astype("int8")
display(bin_int)
display(bin_int.dtype)

,0
0,True
1,False
2,True
3,False
4,True


dtype('bool')

,0
0,1
1,0
2,1
3,0
4,1


dtype('int8')

### Exercício Prático 4
Crie uma coluna `status` em `df_demo` contendo ativo ou inativo. Escolha como quiser.

Crie colunas novas `status_int` e `status_bool` e Converta a coluna com valores 'ativo'/'inativo' para `bool` e para `int8`.  


In [17]:
df_demo['status'] = pd.Series(['ativo','ativo','inativo'])

df_demo['status_int'] = df_demo['status'].map({'ativo' : 1, 'inativo' : 0})
df_demo['status_bool'] = df_demo['status_int'].astype('bool')
df_demo['ativo_int'] = df_demo['ativo'].astype('int8')

df_demo.head()

,produto,categoria,qtd,preco,ativo,data_venda,status,status_int,status_bool,ativo_int
0,A,bronze,10,12.5,True,2025-08-10,ativo,1,True,1
1,B,prata,5,20.0,True,2025-08-11,ativo,1,True,1
2,C,ouro,8,12.5,False,2025-08-12,inativo,0,False,0


## 5) Manipulação de Data/Hora com `pandas`
- `pd.to_datetime` faz parsing de strings para `datetime64[ns]`.
- Atributos `.dt`: `year`, `month`, `day`, `day_name()`, `week`, `quarter`, etc.
- Operações: diferença de datas (`Timedelta`), **re amostragem** (*resample*), janelas móveis (*rolling*).
- Fuso horário: `dt.tz_localize`, `dt.tz_convert` (avançado).

> Sempre validar o formato de entrada (`format=`) e usar `errors='coerce'` para tratar casos ruins.

In [18]:
datas = pd.Series(["10/08/2025", "11/08/2025", "12/08/2025", "13/08/2025"])
dt = pd.to_datetime(datas, dayfirst=True, errors="coerce")
display(dt)

# Extraindo componentes
df_tempo = pd.DataFrame({"data": dt}).dropna()
df_tempo["ano"] = df_tempo["data"].dt.year
df_tempo["mes"] = df_tempo["data"].dt.month
df_tempo["dia"] = df_tempo["data"].dt.day
#df_tempo["dia_semana"] = df_tempo["data"].dt.day_name()
df_tempo["dia_semana"] = df_tempo["data"].dt.dayofweek.map({0: 'Segunda-feira', 1: 'Terça-feira', 2: 'Quarta-feira', 3: 'Quinta-feira', 4: 'Sexta-feira', 5: 'Sábado', 6: 'Domingo'})
display(df_tempo)

# Diferença entre datas
df_tempo["delta_dias"] = (df_tempo["data"].max() - df_tempo["data"]).dt.days
display(df_tempo)

,0
0,2025-08-10
1,2025-08-11
2,2025-08-12
3,2025-08-13


,data,ano,mes,dia,dia_semana
0,2025-08-10,2025,8,10,Domingo
1,2025-08-11,2025,8,11,Segunda-feira
2,2025-08-12,2025,8,12,Terça-feira
3,2025-08-13,2025,8,13,Quarta-feira


,data,ano,mes,dia,dia_semana,delta_dias
0,2025-08-10,2025,8,10,Domingo,3
1,2025-08-11,2025,8,11,Segunda-feira,2
2,2025-08-12,2025,8,12,Terça-feira,1
3,2025-08-13,2025,8,13,Quarta-feira,0


### Exercício Prático 5
Crie um `DataFrame` com uma coluna de datas (período diário de 30 dias começando hoje).  
1. Crie `ano`, `mes`, `dia_da_semana`.  
2. Marque `é_fim_de_semana` (`True` para sábado/domingo).  



## 6) Criação e Transformação de Variáveis (Feature Engineering)
- Operações aritméticas: combinações e razões (*ratios*).
- Condicionais: `np.where`, `.mask`, `.clip`.
- Texto: `.str.lower()`, `.str.contains()`, `.str.extract()`.
- *Binning*: `pd.cut` (intervalos fixos) e `pd.qcut` (quantis).
- Substituições: `.map`, `.replace`, `fillna`.

In [19]:
df_eng = df_demo.copy()

# 1) Condicional: se qtd >= 10, dar 5% off no preço
df_eng["preco_com_desconto"] = np.where(df_eng["qtd"] >= 10, df_eng["preco"] * 0.95, df_eng["preco"])

# 2) Binning de preço
df_eng["faixa_preco"] = pd.cut(df_eng["preco"], bins=[0,15,30,100], labels=["baixo","médio","alto"], include_lowest=True)

# 3) String ops: tag
df_eng["tag"] = df_eng["produto"].str.lower().str.cat(df_eng["categoria"].astype(str), sep="_")

print("Original:"); display(df_demo)
print("Alterada:"); display(df_eng)

Original:


,produto,categoria,qtd,preco,ativo,data_venda,status,status_int,status_bool,ativo_int
0,A,bronze,10,12.5,True,2025-08-10,ativo,1,True,1
1,B,prata,5,20.0,True,2025-08-11,ativo,1,True,1
2,C,ouro,8,12.5,False,2025-08-12,inativo,0,False,0


Alterada:


,produto,categoria,qtd,preco,ativo,data_venda,status,status_int,status_bool,ativo_int,preco_com_desconto,faixa_preco,tag
0,A,bronze,10,12.5,True,2025-08-10,ativo,1,True,1,11.875,baixo,a_bronze
1,B,prata,5,20.0,True,2025-08-11,ativo,1,True,1,20.000,médio,b_prata
2,C,ouro,8,12.5,False,2025-08-12,inativo,0,False,0,12.500,baixo,c_ouro


### Exercício Prático 6
Dado `df_demo`:
1. Crie `receita` (qtd × preco) e `receita_log`.  
2. Crie `faixa_receita` com `pd.qcut` em 3 grupos (tercis).  
3. Gere `promo` = `True` se `categoria` for `ouro` **ou** `qtd >= 10`. Explique o raciocínio.

## 7) Conversão entre Tipos de Dados
- `astype`: conversões diretas (`int32`, `float32`, `category`, `string[python]`, `boolean`).
- `pd.to_numeric`, `pd.to_datetime`, `pd.to_timedelta` (com `errors='coerce'` para valores inválidos).
- Categóricas: `pd.Categorical` (ordem, categorias explicitadas).
- Boas práticas: **nunca** force `int` se houver `NaN` (use `Int64`/`Int32` *nullable*).

> **Regra de ouro**: converta com cuidado, valide com `isna().sum()` e `value_counts(dropna=False)`.

In [20]:
df_conv = pd.DataFrame({
    "preco_str": ["12,50","20","12.5","R$ 50,00","vinte"],
    "data_str": ["2025-08-01","2025-08-02","2025-08-03","2025-08-04","ontem"],
    "flag_str": ["Sim","Não","Sim","Sim","Talvez"],
})

# 1) Preço -> número (normalizando vírgula e removendo símbolos)
tmp = df_conv["preco_str"].str.replace("R$","", regex=False).str.strip()
tmp = tmp.str.replace(",",".", regex=False)
df_conv["preco_num"] = pd.to_numeric(tmp, errors="coerce")

# 2) Datas robustas
df_conv["data_dt"] = pd.to_datetime(df_conv["data_str"], errors="coerce")

# 3) Flag -> bool (map seguro)
df_conv["flag_bool"] = df_conv["flag_str"].map({"Sim": True, "Não": False}).astype("boolean")

display(df_conv)
display(df_conv.dtypes)

,preco_str,data_str,flag_str,preco_num,data_dt,flag_bool
0,"12,50",2025-08-01,Sim,12.5,2025-08-01,True
1,20,2025-08-02,Não,20.0,2025-08-02,False
2,12.5,2025-08-03,Sim,12.5,2025-08-03,True
3,"R$ 50,00",2025-08-04,Sim,50.0,2025-08-04,True
4,vinte,ontem,Talvez,NaN,NaT,<NA>


,0
preco_str,object
data_str,object
flag_str,object
preco_num,float64
data_dt,datetime64[ns]
flag_bool,boolean


### Exercício Prático 7
A partir de uma coluna `texto_valor` contendo valores como `"1.234,56"`, `"999"`, `"R$ 10,00"` e `"n/a"`, crie `valor_float` seguro e documente os passos (limpeza, substituições, `to_numeric`).  

## 8) Boas Práticas e *Checklist* de Tipagem
- Identifique tipos logo ao ler os dados (`read_csv` com `dtype=` quando possível).
- Para categóricas frequentes, use `category` (memória!).
- Sempre trate datas com `to_datetime` + validação.
- Use `errors='coerce'` para detectar lixo e **trate os `NaN` conscientemente**.
- Documente **assunções de negócio** ao criar variáveis derivadas.